In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def hough_circle_detection(image_path, 
                           dp=1.2,
                           min_dist=20,
                           param1=50,
                           param2=30,
                           min_radius=10,
                           max_radius=100,
                           preprocess=False,
                           use_canny=True,
                           show_info=False,
                           show_value=False):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Не удалось загрузить изображение {image_path}")
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if preprocess:
        blurred = cv2.GaussianBlur(gray, (9, 9), 2)
        if use_canny:
            edges = cv2.Canny(blurred, param1, param2)
            processed_image = edges
        else:
            processed_image = blurred
    else:
        processed_image = gray

    circles = cv2.HoughCircles(processed_image, 
                               cv2.HOUGH_GRADIENT, dp=dp, minDist=min_dist, 
                               param1=param1, param2=param2, 
                               minRadius=min_radius, maxRadius=max_radius)

    output = image.copy()
    circles_only_image = np.zeros_like(image)

    if circles is not None:
        circles = np.uint16(np.around(circles))
        for circle in circles[0, :]:
            x, y, r = circle
            cv2.circle(output, (x, y), r, (0, 0, 255), 2)
            cv2.circle(output, (x, y), 2, (0, 0, 255), 3)
            cv2.circle(circles_only_image, (x, y), r, (0, 0, 255), 2)

    titles = ['Исходное изображение', 
              'Предобработанное изображение' if preprocess else 'Серое изображение', 
              'Обнаруженные окружности', 
              'Только окружности']
    images = [image, processed_image, output, circles_only_image]

    plt.figure(figsize=(15, 10))
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(2, 2, i + 1)
        if i == 1:
            plt.imshow(img, cmap='gray')
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
    if show_info:
        if circles is not None:
            print(f"Найдено яиц: {len(circles[0])}")
            for circle in circles[0]:
                x, y, r = circle
                print(f"Центр: ({x}, {y}), радиус: {r}")
        else:
            print("Яйца не найдены")

        return output
    if show_value:
        if circles is not None:
            print(f"Найдено яиц: {len(circles[0])}")
        else:
            print("Яйца не найдены")

hough_circle_detection('../examples/input.jpg', 
                       dp=1.4, min_dist=20, 
                       param1=300, param2=25, 
                       min_radius=10, max_radius=50, 
                       preprocess=True, use_canny=True, show_value=True)